<a href="https://colab.research.google.com/github/TraizeCPerin/Final-Project-CS2/blob/main/Rosal_Kinematics_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget -O kinematics_2d.json "https://raw.githubusercontent.com/TraizeCPerin/Final-Project-CS2/main/data/kinematics_2d.json"

--2026-03-03 11:24:40--  https://raw.githubusercontent.com/TraizeCPerin/Final-Project-CS2/main/data/kinematics_2d.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1837 (1.8K) [text/plain]
Saving to: ‘kinematics_2d.json’

kinematics_2d.json  100%[===================>]   1.79K  --.-KB/s    in 0s      

2026-03-03 11:24:40 (25.8 MB/s) - ‘kinematics_2d.json’ saved [1837/1837]



In [ ]:

!pip install firebase-admin

import json
import math
import firebase_admin
from firebase_admin import credentials, db


with open("kinematics_2d.json") as f:
    data = json.load(f)


cred = credentials.Certificate("firebase_key.json")


if not firebase_admin._apps:
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://rosal-kinematics-db-default-rtdb.asia-southeast1.firebasedatabase.app/'
    })

ref = db.reference("/projectiles")


def max_height(v0, angle_deg, g):
    angle_rad = math.radians(angle_deg)
    return (v0**2 * (math.sin(angle_rad)**2)) / (2 * abs(g))

def horizontal_range(v0, angle_deg, g):
    angle_rad = math.radians(angle_deg)
    return (v0**2 * math.sin(2*angle_rad)) / abs(g)


for scenario in data:
    v0 = scenario["initial_conditions"]["v0"]
    angle = scenario["initial_conditions"]["angle_deg"]
    g = scenario["initial_conditions"]["g"]

    h_max = max_height(v0, angle, g)
    x_range = horizontal_range(v0, angle, g)

    scenario_result = {
        "scenario_id": scenario["scenario_id"],
        "max_height": round(h_max, 2),
        "horizontal_range": round(x_range, 2)
    }


    ref.child(f"scenario_{scenario['scenario_id']}").set(scenario_result)


print("===== JSON Scenarios Pushed to Firebase =====")
for scenario in data:
    v0 = scenario["initial_conditions"]["v0"]
    angle = scenario["initial_conditions"]["angle_deg"]
    g = scenario["initial_conditions"]["g"]

    h_max = max_height(v0, angle, g)
    x_range = horizontal_range(v0, angle, g)

    print(f"Scenario {scenario['scenario_id']} → Max Height: {round(h_max,2)} m, Horizontal Range: {round(x_range,2)} m")

===== JSON Scenarios Pushed to Firebase =====
Scenario 1 → Max Height: 10.19 m, Horizontal Range: 40.77 m
Scenario 2 → Max Height: 2.87 m, Horizontal Range: 19.86 m
Scenario 3 → Max Height: 23.89 m, Horizontal Range: 55.17 m


In [1]:
import math
import firebase_admin
from firebase_admin import credentials, db

# Firebase setup
cred = credentials.Certificate("firebase_key.json")
if not firebase_admin._apps:
    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://rosal-kinematics-db-default-rtdb.asia-southeast1.firebasedatabase.app/'
    })

ref = db.reference("/projectiles")

# Projectile functions
def max_height(v0, angle_deg, g):
    angle_rad = math.radians(angle_deg)
    return (v0**2 * (math.sin(angle_rad)**2)) / (2 * abs(g))

def horizontal_range(v0, angle_deg, g):
    angle_rad = math.radians(angle_deg)
    return (v0**2 * math.sin(2*angle_rad)) / abs(g)

while True:
    print("\n===== SCENARIO MENU =====")
    print("1. Display Scenarios")
    print("2. Add Scenario")
    print("3. Update Scenario")
    print("4. Delete Scenario")
    print("5. Exit")

    choice = input("Enter your choice: ")

    # DISPLAY
    if choice == "1":
        scenarios = ref.get()
        if scenarios:
            print("\nAll Scenarios:")
            for key, value in scenarios.items():
                print(f"Scenario {value['scenario_id']} → Max Height: {value['max_height']} m, Horizontal Range: {value['horizontal_range']} m")
        else:
            print("No scenarios found.")

    # ADD
    elif choice == "2":
        scenario_id = input("Enter Scenario ID: ")
        v0 = float(input("Enter Initial Velocity (m/s): "))
        angle = float(input("Enter Angle (degrees): "))
        g = float(input("Enter Gravity (m/s^2): "))

        h_max = max_height(v0, angle, g)
        x_range = horizontal_range(v0, angle, g)

        ref.child(f"scenario_{scenario_id}").set({
            "scenario_id": int(scenario_id),
            "max_height": round(h_max, 2),
            "horizontal_range": round(x_range, 2)
        })
        print(f"Scenario {scenario_id} added successfully!")

    # UPDATE
    elif choice == "3":
        scenario_id = input("Enter Scenario ID to update: ")
        scenario = ref.child(f"scenario_{scenario_id}").get()
        if scenario:
            v0 = float(input("Enter new Initial Velocity (m/s): "))
            angle = float(input("Enter new Angle (degrees): "))
            g = float(input("Enter new Gravity (m/s^2): "))

            h_max = max_height(v0, angle, g)
            x_range = horizontal_range(v0, angle, g)

            ref.child(f"scenario_{scenario_id}").update({
                "max_height": round(h_max, 2),
                "horizontal_range": round(x_range, 2)
            })
            print(f"Scenario {scenario_id} updated successfully!")
        else:
            print("Scenario not found.")

    # DELETE
    elif choice == "4":
        scenario_id = input("Enter Scenario ID to delete: ")
        if ref.child(f"scenario_{scenario_id}").get():
            ref.child(f"scenario_{scenario_id}").delete()
            print(f"Scenario {scenario_id} deleted successfully!")
        else:
            print("Scenario not found.")

    # EXIT
    elif choice == "5":
        print("Exiting menu...")
        break

    else:
        print("Invalid choice. Enter 1-5.")


===== SCENARIO MENU =====
1. Display Scenarios
2. Add Scenario
3. Update Scenario
4. Delete Scenario
5. Exit
Enter your choice: 1

All Scenarios:
Scenario 1 → Max Height: 10.19 m, Horizontal Range: 40.77 m
Scenario 2 → Max Height: 2.87 m, Horizontal Range: 19.86 m
Scenario 3 → Max Height: 23.89 m, Horizontal Range: 55.17 m
Scenario 5 → Max Height: 0.0 m, Horizontal Range: 0.1 m

===== SCENARIO MENU =====
1. Display Scenarios
2. Add Scenario
3. Update Scenario
4. Delete Scenario
5. Exit
Enter your choice: 2
Enter Scenario ID: 6
Enter Initial Velocity (m/s): 8
Enter Angle (degrees): 9
Enter Gravity (m/s^2): 5
Scenario 6 added successfully!

===== SCENARIO MENU =====
1. Display Scenarios
2. Add Scenario
3. Update Scenario
4. Delete Scenario
5. Exit
Enter your choice: 1

All Scenarios:
Scenario 1 → Max Height: 10.19 m, Horizontal Range: 40.77 m
Scenario 2 → Max Height: 2.87 m, Horizontal Range: 19.86 m
Scenario 3 → Max Height: 23.89 m, Horizontal Range: 55.17 m
Scenario 5 → Max Height: 0.